In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np
from sklearn.naive_bayes import MultinomialNB
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
df = df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, errors='ignore')

In [ ]:
y = df['v1']
X = df['v2']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
vectorizer_demo = CountVectorizer(max_features=10)
X_train_demo = vectorizer_demo.fit_transform(X_train)

In [ ]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

In [ ]:
pipeline = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('classifier', MultinomialNB())
])

In [ ]:
param_grid = {

    'vectorizer__max_features': [1000, 3000, 5000],
    'vectorizer__min_df': [1, 2],
    'vectorizer__ngram_range': [(1, 1), (1, 2)],
    'classifier__alpha': [0.1, 0.5, 1.0, 2.0],
    'classifier__fit_prior': [True, False]
}

In [ ]:
total_combinations = (len(param_grid['vectorizer__max_features']) *
                     len(param_grid['vectorizer__min_df']) *
                     len(param_grid['vectorizer__ngram_range']) *
                     len(param_grid['classifier__alpha']) *
                     len(param_grid['classifier__fit_prior']))

print(f"\nTotal de combinações: {total_combinations}")


In [ ]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
y_pred = grid_search.predict(X_test)

In [ ]:
test_accuracy = accuracy_score(y_test, y_pred)

print(f"\nDesempenho do modelo:")
print(f"  Acurácia no teste: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"  Acurácia na validação cruzada: {grid_search.best_score_:.4f} ({grid_search.best_score_*100:.2f}%)")

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['ham', 'spam'])

print(f"\nMatriz de Confusão:")
print(cm)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Valores absolutos
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham', 'Spam'])
disp1.plot(ax=axes[0], cmap='RdPu', values_format='d', colorbar=True)
axes[0].set_title('matriz de confusão 1')
axes[0].grid(False)

# Valores normalizados
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=['Ham', 'Spam'])
disp2.plot(ax=axes[1], cmap='Oranges', values_format='.2%', colorbar=True)
axes[1].set_title('matriz de confusão 2')
axes[1].grid(False)

In [ ]:
report = classification_report(y_test, y_pred, target_names=['Ham', 'Spam'])
print(report)

precision = precision_score(y_test, y_pred, average=None, labels=['ham', 'spam'])
recall = recall_score(y_test, y_pred, average=None, labels=['ham', 'spam'])
f1 = f1_score(y_test, y_pred, average=None, labels=['ham', 'spam'])
